# Trial Eligbility

**Assess number of patients in the mCRC analytic cohort meeting KEYNOTE-177 ECOG and lab eligbility.**

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorColorectal, merge_dataframes

## Import dfs

In [2]:
index_df = pd.read_csv('../outputs/pembro_chemo_index.csv')

In [3]:
index_df.shape

(26405, 3)

In [4]:
# remove time portion and leave just YYYY-MM-DD
index_df['StartDate'] = index_df['StartDate'].astype(str).str.split(' ').str[0]

In [5]:
dtype_map = pd.read_csv('../outputs/pembro_chemo_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/pembro_chemo_features_df.csv', dtype = dtype_map)

In [6]:
features_df.shape

(1208, 170)

In [7]:
main_df = pd.merge(features_df, index_df, on = 'PatientID', how = 'left')

In [8]:
main_df = main_df.query('met_diagnosis_year <= 2023')

In [9]:
main_df.shape

(1074, 172)

In [10]:
main_df = main_df[['PatientID', 'LineName', 'StartDate']]

In [11]:
main_df.head(3)

,PatientID,LineName,StartDate
0,F12B04C749133,chemo,2019-02-20
1,FFFB7D51F655C,pembro,2023-10-04
2,F6E393E2A3FBA,chemo,2019-05-13


## ECOG and Labs

In [12]:
# Initialize class 
processor = DataProcessorColorectal()

In [13]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = main_df,
                                 index_date_column = 'StartDate',
                                 additional_loinc_mappings = {'neutrophil': ['26499-4', '751-8', '753-4']},
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-04-25 23:02:59,644 - INFO - Successfully read Lab.csv file with shape: (47135286, 17) and unique PatientIDs: 44960
2026-04-25 23:03:14,927 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (1438151, 18) and unique PatientIDs: 1061
2026-04-25 23:03:18,123 - INFO - Successfully processed Lab.csv file with final shape: (1074, 81) and unique PatientIDs: 1074


In [14]:
labs_df = labs_df[['PatientID', 'neutrophil', 'hemoglobin', 'platelet', 'creatinine', 'total_bilirubin', 'ast', 'alt', 'alp', 'albumin']]

In [15]:
labs_df.shape

(1074, 10)

In [16]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = main_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-04-25 23:03:18,595 - INFO - Successfully read ECOG.csv file with shape: (1017535, 4) and unique PatientIDs: 36246
2026-04-25 23:03:18,737 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (31623, 5) and unique PatientIDs: 910
2026-04-25 23:03:18,757 - INFO - Successfully processed ECOG.csv file with final shape: (1074, 3) and unique PatientIDs: 1074


In [17]:
ecog_df = ecog_df[['PatientID', 'ecog_index']]

In [18]:
ecog_df.shape

(1074, 2)

In [19]:
main_df = pd.merge(main_df, labs_df, on = 'PatientID', how = 'left')

In [20]:
main_df = pd.merge(main_df, ecog_df, on = 'PatientID', how = 'left')

In [21]:
main_df.shape

(1074, 13)

In [22]:
main_df.head(3)

,PatientID,LineName,StartDate,neutrophil,hemoglobin,platelet,creatinine,total_bilirubin,ast,alt,alp,albumin,ecog_index
0,F12B04C749133,chemo,2019-02-20,NaN,10.4,322.0,1.32,1.3,28.0,25.0,225.0,43.0,NaN
1,FFFB7D51F655C,pembro,2023-10-04,3.90,12.5,312.0,0.91,0.4,14.0,21.0,51.0,42.0,0
2,F6E393E2A3FBA,chemo,2019-05-13,3.43,15.2,232.0,0.83,0.2,23.0,47.0,93.0,45.0,0


## ECOG Eligibility

In [23]:
main_df['ecog_eligible'] = np.where((main_df['ecog_index'] <= 1), 1, 0)

In [24]:
main_df['ecog_ineligible'] = np.where(main_df['ecog_index'] > 1, 1, 0)

In [25]:
main_df['ecog_indeterminate'] = np.where(main_df['ecog_index'].isna(), 1, 0)

In [26]:
main_df.shape[0]

1074

In [27]:
main_df.ecog_eligible.value_counts()[1]+main_df.ecog_ineligible.value_counts()[1]+main_df.ecog_indeterminate.value_counts()[1]

np.int64(1074)

## Lab eligibility

In [30]:
main_df['lab_eligible'] = np.where(
    (main_df['neutrophil'] >= 1.5) &
    (main_df['platelet'] >= 100) &
    (main_df['hemoglobin'] >= 9) &
    (main_df['creatinine'] <= 1.8) & 
    (main_df['total_bilirubin'] <= 1.8) &
    (main_df['ast'] <= 100) &
    (main_df['alt'] <= 100) &
    (main_df['alp'] <= 300) & 
    (main_df['albumin'] >= 25), 1, 0)

In [31]:
main_df['lab_ineligible'] = np.where(
    (main_df['neutrophil'] < 1.5) |
    (main_df['platelet'] < 100) |
    (main_df['hemoglobin'] < 9) |
    (main_df['creatinine'] > 1.8) | 
    (main_df['total_bilirubin'] > 1.8) |
    (main_df['ast'] > 100) |
    (main_df['alt'] > 100) |
    (main_df['alp'] > 300) | 
    (main_df['albumin'] < 25), 1, 0)

In [32]:
main_df['lab_indeterminate'] = np.where((main_df['lab_eligible'] == 0) & (main_df['lab_ineligible'] == 0), 1, 0)

In [33]:
main_df.lab_eligible.value_counts()[1]+main_df.lab_ineligible.value_counts()[1]+main_df.lab_indeterminate.value_counts()[1]

np.int64(1074)

## Overal eligibility

In [34]:
main_df['eligible'] = np.where((main_df['ecog_eligible'] == 1) & (main_df['lab_eligible'] == 1), 1, 0)

In [35]:
main_df['ineligible'] = np.where((main_df['ecog_ineligible'] == 1) | (main_df['lab_ineligible'] == 1), 1, 0)

In [36]:
main_df['indeterminate'] = np.where((main_df['eligible'] == 0) & (main_df['ineligible'] == 0), 1, 0)

In [37]:
print('Overall cohort')
print(f'Eligible: count {main_df.eligible.value_counts()[1]}, percentage {round(main_df.eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.ineligible.value_counts()[1]}, percentage {round(main_df.ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.indeterminate.value_counts()[1]}, percentage {round(main_df.indeterminate.value_counts(normalize = True)[1]*100, 2)}')

Overall cohort
Eligible: count 404, percentage 37.62
Ineligible: count 269, percentage 25.05
Indeterminate: count 401, percentage 37.34
